# Banking Transaction Fraud & Risk Analytics — Exploratory Data Analysis

**Dataset:** Bank Transaction Dataset for Fraud Detection (Kaggle)  
**Method:** Isolation Forest Anomaly Detection (no fraud label available)  

This notebook walks through:
1. Dataset inspection
2. Data cleaning
3. Feature engineering
4. Exploratory visualisations
5. Anomaly detection & risk scoring
6. Explainability analysis

## 0. Setup & Imports

In [ ]:
import os
import sys
import json
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# Add src to path so we can import our modules
sys.path.insert(0, os.path.join('..', 'src'))

from data_preprocessing import load_raw_data, inspect_data, clean_data
from feature_engineering import build_features, MODEL_FEATURES
from anomaly_detection import train, score_transactions

sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.figsize'] = (12, 5)

print('Libraries loaded successfully.')

## 1. Load & Inspect Dataset

In [ ]:
RAW_PATH = os.path.join('..', 'data', 'raw', 'bank_transactions_data_2.csv')
raw_df   = load_raw_data(RAW_PATH)
raw_df.head()

In [ ]:
summary = inspect_data(raw_df)
print(f"Shape     : {summary['n_rows']:,} rows × {summary['n_cols']} columns")
print(f"Duplicates: {summary['n_duplicates']}")
print()
print("=== Column dtypes ===")
print(pd.Series(summary['dtypes']))
print()
print("=== Missing values ===")
print(pd.Series(summary['missing_values']))

In [ ]:
print("=== Unique value counts ===")
print(pd.Series(summary['unique_counts']))

In [ ]:
raw_df.describe()

## 2. Data Cleaning

In [ ]:
clean_df = clean_data(raw_df)
print(f'Rows after cleaning: {len(clean_df):,}')
print(f'Columns: {list(clean_df.columns)}')

In [ ]:
# Categorical column value counts
for col in ['TransactionType', 'Channel', 'CustomerOccupation', 'Location']:
    print(f'\n{col}:')
    print(clean_df[col].value_counts())

## 3. Feature Engineering

In [ ]:
df = build_features(clean_df)
print(f'Shape after feature engineering: {df.shape}')
print(f'\nNew feature columns: {[c for c in df.columns if c not in clean_df.columns]}')

## 4. Exploratory Data Analysis

### 4.1 Transaction Amount Distribution

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].hist(df['TransactionAmount'], bins=50, color='steelblue', edgecolor='white')
axes[0].set_title('Transaction Amount Distribution')
axes[0].set_xlabel('Amount ($)')
axes[0].set_ylabel('Count')

df.boxplot(column='TransactionAmount', ax=axes[1])
axes[1].set_title('Transaction Amount Box Plot')
axes[1].set_ylabel('Amount ($)')

plt.tight_layout()
plt.show()

print(df['TransactionAmount'].describe())

### 4.2 Transaction Type (Credit vs Debit)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

type_counts = df['TransactionType'].value_counts()
axes[0].pie(type_counts.values, labels=type_counts.index, autopct='%1.1f%%',
            colors=['#3b82f6', '#22c55e'])
axes[0].set_title('Transaction Type')

type_amount = df.groupby('TransactionType')['TransactionAmount'].mean()
axes[1].bar(type_amount.index, type_amount.values, color=['#3b82f6', '#22c55e'])
axes[1].set_title('Average Amount by Type')
axes[1].set_ylabel('Avg Amount ($)')

plt.tight_layout()
plt.show()

### 4.3 Channel Distribution

In [ ]:
channel_counts = df['Channel'].value_counts()
fig, ax = plt.subplots(figsize=(8, 4))
ax.bar(channel_counts.index, channel_counts.values, color=['#3b82f6', '#f59e0b', '#22c55e'])
ax.set_title('Transactions by Channel')
ax.set_ylabel('Count')
for i, v in enumerate(channel_counts.values):
    ax.text(i, v + 5, str(v), ha='center')
plt.tight_layout()
plt.show()

### 4.4 Top Locations

In [ ]:
top_locs = df['Location'].value_counts().head(15)
fig, ax  = plt.subplots(figsize=(10, 6))
top_locs.sort_values().plot(kind='barh', ax=ax, color='steelblue')
ax.set_title('Top 15 Transaction Locations')
ax.set_xlabel('Number of Transactions')
plt.tight_layout()
plt.show()

### 4.5 Customer Occupation & Age

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

occ_counts = df['CustomerOccupation'].value_counts()
axes[0].bar(occ_counts.index, occ_counts.values, color=sns.color_palette('pastel'))
axes[0].set_title('Transactions by Occupation')
axes[0].set_ylabel('Count')
axes[0].tick_params(axis='x', rotation=15)

axes[1].hist(df['CustomerAge'], bins=30, color='coral', edgecolor='white')
axes[1].set_title('Customer Age Distribution')
axes[1].set_xlabel('Age')

plt.tight_layout()
plt.show()

### 4.6 Transaction Trends Over Time

In [ ]:
daily = df.set_index('TransactionDate').resample('W')['TransactionAmount'].agg(['count', 'sum'])
daily.columns = ['count', 'total_amount']

fig, axes = plt.subplots(2, 1, figsize=(14, 8))

axes[0].plot(daily.index, daily['count'], color='steelblue', linewidth=2)
axes[0].set_title('Weekly Transaction Count')
axes[0].set_ylabel('Transactions')

axes[1].plot(daily.index, daily['total_amount'], color='coral', linewidth=2)
axes[1].set_title('Weekly Transaction Volume ($)')
axes[1].set_ylabel('Total Amount ($)')

plt.tight_layout()
plt.show()

### 4.7 Login Attempts Analysis

In [ ]:
login_counts = df['LoginAttempts'].value_counts().sort_index()

fig, ax = plt.subplots(figsize=(8, 4))
ax.bar(login_counts.index.astype(str), login_counts.values, color='#f59e0b')
ax.set_title('Distribution of Login Attempts')
ax.set_xlabel('Login Attempts')
ax.set_ylabel('Transaction Count')

for i, v in enumerate(login_counts.values):
    ax.text(i, v + 2, str(v), ha='center')

plt.tight_layout()
plt.show()

print(f"Transactions with >3 login attempts: {(df['LoginAttempts'] > 3).sum()} "
      f"({(df['LoginAttempts'] > 3).mean()*100:.1f}%)")

### 4.8 Account Balance Analysis

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].hist(df['AccountBalance'], bins=50, color='purple', edgecolor='white', alpha=0.7)
axes[0].set_title('Account Balance Distribution')
axes[0].set_xlabel('Balance ($)')

axes[1].scatter(df['AccountBalance'], df['TransactionAmount'],
                alpha=0.3, color='steelblue', s=15)
axes[1].set_title('Balance vs Transaction Amount')
axes[1].set_xlabel('Account Balance ($)')
axes[1].set_ylabel('Transaction Amount ($)')

plt.tight_layout()
plt.show()

print(df['AccountBalance'].describe())

### 4.9 Transaction Duration

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].hist(df['TransactionDuration'], bins=40, color='green', edgecolor='white', alpha=0.7)
axes[0].set_title('Transaction Duration Distribution')
axes[0].set_xlabel('Duration (s)')

df.boxplot(column='TransactionDuration', by='Channel', ax=axes[1])
axes[1].set_title('Duration by Channel')
axes[1].set_xlabel('Channel')
axes[1].set_ylabel('Duration (s)')

plt.tight_layout()
plt.show()

### 4.10 Correlation Matrix

In [ ]:
num_cols = ['TransactionAmount', 'AccountBalance', 'TransactionDuration',
            'LoginAttempts', 'CustomerAge', 'hours_since_prev_tx',
            'amount_to_balance_ratio', 'account_tx_count', 'account_n_devices']

corr = df[num_cols].corr()

plt.figure(figsize=(12, 8))
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=True, fmt='.2f', cmap='RdBu_r',
            center=0, square=True, linewidths=0.5)
plt.title('Correlation Matrix — Key Numeric Features')
plt.tight_layout()
plt.show()

## 5. Anomaly Detection — Train & Score

In [ ]:
# Train Isolation Forest
pipeline = train(df, MODEL_FEATURES)
print('Model trained.')

In [ ]:
# Score all transactions
scored_df = score_transactions(pipeline, df, MODEL_FEATURES)
scored_df[['TransactionID', 'TransactionAmount', 'risk_score_pct', 'risk_level']].head(10)

In [ ]:
# Risk level distribution
risk_counts = scored_df['risk_level'].value_counts()
print(risk_counts)

fig, ax = plt.subplots(figsize=(7, 4))
colors = {'High': '#ef4444', 'Medium': '#f59e0b', 'Low': '#22c55e'}
for level, count in risk_counts.items():
    ax.bar(level, count, color=colors.get(level, 'gray'))
    ax.text(level, count + 2, str(count), ha='center')
ax.set_title('Risk Level Distribution')
ax.set_ylabel('Transactions')
plt.tight_layout()
plt.show()

In [ ]:
# Risk score distribution
fig, ax = plt.subplots(figsize=(10, 4))
for level, color in colors.items():
    subset = scored_df[scored_df['risk_level'] == level]
    ax.hist(subset['risk_score_pct'], bins=30, alpha=0.7, label=level, color=color)
ax.set_title('Risk Score Distribution by Risk Level')
ax.set_xlabel('Risk Score (%)')
ax.set_ylabel('Count')
ax.legend()
plt.tight_layout()
plt.show()

## 6. High-Risk Transaction Analysis

In [ ]:
high_risk = scored_df[scored_df['risk_level'] == 'High']
low_risk  = scored_df[scored_df['risk_level'] == 'Low']

comparison = pd.DataFrame({
    'Feature': ['TransactionAmount', 'LoginAttempts', 'TransactionDuration',
                'amount_to_balance_ratio', 'account_n_devices'],
    'High Risk Avg': [
        high_risk['TransactionAmount'].mean(),
        high_risk['LoginAttempts'].mean(),
        high_risk['TransactionDuration'].mean(),
        high_risk['amount_to_balance_ratio'].mean(),
        high_risk['account_n_devices'].mean(),
    ],
    'Low Risk Avg': [
        low_risk['TransactionAmount'].mean(),
        low_risk['LoginAttempts'].mean(),
        low_risk['TransactionDuration'].mean(),
        low_risk['amount_to_balance_ratio'].mean(),
        low_risk['account_n_devices'].mean(),
    ],
})
comparison['Ratio (High/Low)'] = (comparison['High Risk Avg'] / comparison['Low Risk Avg']).round(2)
comparison.round(3)

In [ ]:
# Top accounts by high risk count
scored_df[scored_df['risk_level'] == 'High'].groupby('AccountID').size().sort_values(ascending=False).head(10)

## 7. Feature Importance (Permutation-Based)

In [ ]:
fi_path = os.path.join('..', 'models', 'feature_importances.json')
if os.path.exists(fi_path):
    with open(fi_path) as f:
        fi = json.load(f)
    
    top_fi = dict(list(fi.items())[:15])
    
    fig, ax = plt.subplots(figsize=(10, 6))
    items = sorted(top_fi.items(), key=lambda x: x[1])
    ax.barh([x[0] for x in items], [x[1] for x in items], color='steelblue')
    ax.set_title('Top 15 Feature Importances (Permutation)')
    ax.set_xlabel('Mean Importance')
    plt.tight_layout()
    plt.show()
else:
    print('Feature importances not found. Run model_training.py first.')

## 8. Summary

### Key Findings

- The dataset contains **~2,500 banking transactions** with **no fraud label**.
- Isolation Forest identified approximately **8–15%** of transactions as anomalous.
- High-risk transactions tend to have **higher amounts, more login attempts, and higher amount-to-balance ratios**.
- **ATM and Online channels** show similar transaction volumes.
- Students represent the most active customer segment.
- The distribution of transaction amounts is **right-skewed** with several large outliers.

### Next Steps

1. Launch the Streamlit dashboard: `streamlit run app/app.py`
2. Use the Risk Detection page to investigate individual transactions
3. Use the SQL Explorer for custom analytical queries
4. Review Account Risk Analysis for high-risk account profiles